In [ ]:
# -- B. Génération des Fiches de Vocabulaire & Pages d'Exercices --
for node, data in arborescence.items():
    for fiche in data['fiches']:
        chemin_txt = FILES_DIR / os.path.join(*node) / f"{fiche}.txt"
        segments_fiche = list(node) + [fiche]
        
        # Chemins cibles pour les deux pages HTML
        fichier_html_vocab = construire_chemin_page(segments_fiche, est_fiche=True)
        fichier_html_exos = PAGES_DIR / f"{fichier_html_vocab.stem}_exos.html"
        
        titre_propre = fiche.split(';')[0].replace("-", " ").capitalize() if ';' in fiche else fiche.replace("-", " ").capitalize()
        
        # --- Listes de stockage des données ---
        mots_vocabulaire = [] # Stocke les dicts de mots
        exercices_par_onglet = {"Fiche de révisions": []} # Onglet révision toujours présent
        
        lecture_exercices = False
        lien_quizlet = None
        
        # --- LECTURE DU FICHIER SOURCE .TXT ---
        with open(chemin_txt, \"r\", encoding=\"utf-8\") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                
                # Lien Quizlet global
                if line.startswith("QUIZLET;") or line.startswith("LIEN;") or line.startswith("http"): 
                    parts_lien = line.split(';')
                    lien_quizlet = parts_lien[1].strip() if len(parts_lien) > 1 else parts_lien[0].strip()
                    continue
                    
                if line.startswith("#"):
                    continue
                
                # Détection du basculement vers la section exercices
                if line == "=== EXERCICES ===":
                    lecture_exercices = True
                    continue
                
                if not lecture_exercices:
                    # Lecture Vocabulaire (7 colonnes)
                    parts = [p.strip() for p in line.split(';')]
                    if len(parts) < 7: continue
                    
                    mots_vocabulaire.append({
                        "anglais": parts[0], "francais": parts[1],
                        "note": "" if parts[2] == "..." else parts[2],
                        "exemple": "" if parts[3] == "..." else parts[3],
                        "type": convertir_type(parts[4]),
                        "provenance": convertir_provenance(parts[5]),
                        "niveau": parts[6]
                    })
                else:
                    # Lecture Exercices (énoncé;réponses;indications;explications;num_exo;type_exo;niveau)
                    parts = [p.strip() for p in line.split(';')]
                    if len(parts) < 7: continue
                    
                    nom_onglet = parts[4] # num_exo sert de nom textuel pour l'onglet
                    if nom_onglet not in exercices_par_onglet:
                        exercices_par_onglet[nom_onglet] = []
                        
                    exercices_par_onglet[nom_onglet].append({
                        "enonce": parts[0], "reponses": parts[1],
                        "indication": "" if parts[2] == "..." else parts[2],
                        "explication": "" if parts[3] == "..." else parts[3],
                        "type": parts[5].upper(), "niveau": parts[6]
                    })
        
        # =====================================================================
        # 1. GÉNÉRATION DE LA PAGE DE VOCABULAIRE
        # =====================================================================
        tableau_html = '<table class=\"vocab-table\">\n<thead>\n<tr><th>Type</th><th>Anglais</th><th>Français</th><th>Niveau</th><th>Note</th><th>Exemple</th><th>Provenance</th></tr>\n</thead>\n<tbody>\n'
        for m in mots_vocabulaire:
            type_classe = formater_nom_fichier(m['type'])
            niv_classe = m['niveau'].strip().lower()
            
            cell_type = f"<span class='badge badge-type badge-type-{type_classe}'>{m['type']}</span>" if m['type'] != "..." else ""
            cell_level = f"<span class='badge badge-level badge-level-{niv_classe}'>{m['niveau']}</span>" if m['niveau'] != "..." else ""
            cell_prov = f"<span class='badge badge-prov'>{m['provenance']}</span>" if m['provenance'] != "..." else ""
            
            tableau_html += f"<tr><td>{cell_type}</td><td><span class='txt-anglais'>{m['anglais']}</span></td><td><span class='txt-francais'>{m['francais']}</span></td><td>{cell_level}</td><td><span class='txt-note'>{m['note']}</span></td><td><span class='txt-exemple'>{m['exemple']}</span></td><td>{cell_prov}</td></tr>\n"
        tableau_html += "</tbody>\n</table>"
        
        html_bq = ""
        if lien_quizlet:
            html_bq = f'<a href=\"{lien_quizlet}\" target=\"_blank\" class=\"action-button-link quizlet-btn\" title=\"Ouvrir Quizlet\"><img src=\"../images/quizlet_logo.png\" alt=\"Quizlet\"></a>'
            
        corps_page_vocab = f"""
        <div class="fiche-header-container">
            <div class="fiche-titles">
                <p class="intro-text">Fiche de vocabulaire ({len(mots_vocabulaire)} mots)</p>
                <h3 class="fiche-title">{titre_propre}</h3>
            </div>
            <div class="fiche-actions-inline">{html_bq}</div>
        </div>
        
        {tableau_html}
        
        <div style="text-align: center; margin-top: 40px; margin-bottom: 20px;">
            <a href="./{fichier_html_exos.name}" class="action-button-link exercises-main-btn" style="padding: 15px 30px; font-size: 1.1rem; display: inline-flex; align-items: center; gap: 10px; background-color: #28a745; color: white; text-decoration: none; border-radius: 8px; font-weight: bold; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
                🎯 S'entraîner avec les exercices
            </a>
        </div>
        """
        
        breadcrumbs_vocab = generer_breadcrumbs(segments_fiche, est_fiche=True)
        with open(fichier_html_vocab, "w", encoding="utf-8") as f_out:
            f_out.write(TEMPLATE_CONTENT.format(PAGE_TITLE=titre_propre, MAIN_TITLE="ANGLAIS > Vocabulaire", BREADCRUMBS=breadcrumbs_vocab, CONTENT=corps_page_vocab))
            
        # =====================================================================
        # 2. GÉNÉRATION DE LA PAGE D'EXERCICES
        # =====================================================================
        # Boutons d'onglets
        onglets_html = '<div class="onglets-exos-container" style="display: flex; gap: 10px; margin-bottom: 25px; border-bottom: 2px solid #ddd; padding-bottom: 10px;">'
        for i, nom_onglet in enumerate(exercices_par_onglet.keys()):
            classe_active = "active" if i == 0 else ""
            onglets_html += f'<button class="onglet-btn {classe_active}" onclick="basculerFiche(\'{nom_onglet}\')" style="padding: 10px 20px; border: none; background: #f0f0f0; border-radius: 6px; cursor: pointer; font-weight: bold; transition: 0.2s;">{nom_onglet}</button>'
        onglets_html += '</div>'
        
        # Contenu des fiches
        fiches_contenu_html = '<div class="fiches-exos-wrapper">'
        
        # --- Onglet 1 : Fiche de révisions (Généré en JS pour le côté aléatoire) ---
        fiches_contenu_html += f"""
        <div class="fiche-exercice active" data-nom="Fiche de révisions">
            <p class="intro-text" style="margin-bottom: 20px;">Traduisez le mot manquant (Anglais ou Français choisi aléatoirement).</p>
            <div id="container-revision-dynamique"></div>
            <div style="text-align: center; margin-top: 20px;">
                <button onclick="validerFicheRevision()" class="action-button" style="background: #007bff; color: white; padding: 12px 25px; border: none; border-radius: 6px; font-weight: bold; cursor: pointer;">Valider mes réponses</button>
            </div>
        </div>
        """
        
        # --- Autres Onglets (Exercices écrits dans le fichier .txt) ---
        for nom_onglet, liste_exos in exercices_par_onglet.items():
            if nom_onglet == "Fiche de révisions": continue
            
            fiches_contenu_html += f'<div class="fiche-exercice" data-nom="{nom_onglet}" style="display: none;">'
            
            for idx, exo in enumerate(liste_exos):
                html_actions = f'<div class="actions-exo" style="margin-top: 8px; display: flex; gap: 10px; align-items: center;">'
                if exo['indication']:
                    html_actions += f'<button onclick="afficherIndice(this, \'{exo["indication"]}\')" class="btn-mini-info" style="background:#ffc107; border:none; padding:5px 10px; border-radius:4px; cursor:pointer; font-size:0.85rem;">💡 Indice</button>'
                html_actions += f'<span class="txt-indication" style="display:none; color:#775500; font-style:italic; font-size:0.9rem;"></span>'
                html_actions += f'</div>'
                
                html_correction = f'<div class="correction-exo" data-reponses="{exo["reponses"]}" data-explication="{exo["explication"]}" style="margin-top: 10px; display:none; padding:10px; background:#f8f9fa; border-left:4px solid #6c757d; border-radius:0 4px 4px 0;">'
                html_correction += f'<p style="margin:0; font-weight:bold; color:#28a745;">Réponse attendue : {exo["reponses"].replace(",", " , ")}</p>'
                if exo['explication']:
                    html_correction += f'<p style="margin:5px 0 0 0; font-size:0.9rem; color:#555;">💡 <em>{exo["explication"]}</em></p>'
                html_correction += f'</div>'
                
                if exo['type'] == 'TRAD':
                    fiches_contenu_html += f"""
                    <div class="ex-block type-trad" style="background: white; padding: 15px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); margin-bottom: 20px; border-left: 4px solid #007bff;">
                        <p style="margin: 0 0 10px 0;"><strong>Question {idx+1} (Traduction) :</strong> Traduire en anglais : <span style="font-style: italic; font-weight: 500;">{exo['enonce']}</span></p>
                        <input type="text" class="saisie-trad-longue input-custom-exo" placeholder="Tapez votre traduction ici..." style="width: 100%; max-width: 600px; padding: 10px; border: 2px solid #ccc; border-radius: 6px; font-size: 1rem;" />
                        {html_actions}
                        {html_correction}
                    </div>
                    """
                elif exo['type'] == 'TROU':
                    # Remplacement des [...] par des balises input en HTML
                    enonce_html = exo['enonce'].replace("[…]", '<input type="text" class="saisie-trou-courte input-custom-exo" placeholder="..." style="width: 120px; padding: 4px 8px; border: 2px solid #ccc; border-radius: 4px; text-align: center; font-size: 1rem; margin: 0 4px;" />')
                    enonce_html = enonce_html.replace("[...]", '<input type="text" class="saisie-trou-courte input-custom-exo" placeholder="..." style="width: 120px; padding: 4px 8px; border: 2px solid #ccc; border-radius: 4px; text-align: center; font-size: 1rem; margin: 0 4px;" />')
                    
                    fiches_contenu_html += f"""
                    <div class="ex-block type-trou" style="background: white; padding: 15px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); margin-bottom: 20px; border-left: 4px solid #17a2b8;">
                        <p style="margin: 0 0 10px 0;"><strong>Question {idx+1} (Texte à trous) :</strong> Complétez la phrase :</p>
                        <p style="line-height: 1.8; margin: 5px 0;">{enonce_html}</p>
                        {html_actions}
                        {html_correction}
                    </div>
                    """
            
            if liste_exos:
                fiches_contenu_html += f"""
                <div style="text-align: center; margin-top: 20px;">
                    <button onclick="validerFicheStandard(this)" class="action-button" style="background: #28a745; color: white; padding: 12px 25px; border: none; border-radius: 6px; font-weight: bold; cursor: pointer;">Valider cet exercice</button>
                </div>
                """
            fiches_contenu_html += '</div>'
            
        fiches_contenu_html += '</div>'
        
        # --- SCRIPT JAVASCRIPT GLOBAL POUR LES EXERCICES ---
        # Sérialisation propre des mots pour le JS
        mots_json_bruts = [{"en": m["anglais"], "fr": m["francais"]} for m in mots_vocabulaire]
        import json
        mots_json_string = json.dumps(mots_json_bruts, ensure_ascii=False)
        
        js_code = f"""
        <script>
        const vocabData = {mots_json_string};
        
        // 1. Initialisation de la Fiche de Révision Aléatoire au chargement
        document.addEventListener("DOMContentLoaded", () => {{
            const container = document.getElementById("container-revision-dynamique");
            let html = '<table class="vocab-table" style="width:100%; border-collapse: collapse; margin-top:15px;">';
            html += '<thead><tr style="background:#f1f1f1;"><th style="padding:10px; border:1px solid #ddd;">Anglais</th><th style="padding:10px; border:1px solid #ddd;">Français</th></tr></thead><tbody>';
            
            vocabData.forEach((item, index) => {{
                // Choix aléatoire : true = cacher anglais (saisir anglais), false = cacher français (saisir français)
                const cacherAnglais = Math.random() < 0.5;
                
                html += `<tr class="row-revision" data-en="${{item.en}}" data-fr="${{item.fr}}" data-mode="${{cacherAnglais ? 'en' : 'fr'}}" style="border-bottom:1px solid #ddd;">`;
                
                if (cacherAnglais) {{
                    html += `<td style="padding:8px; border:1px solid #ddd;"><input type="text" class="input-rev" placeholder="Traduire en anglais..." style="width:95%; padding:6px; border:2px solid #ccc; border-radius:4px; font-size:0.95rem;"/></td>`;
                    html += `<td style="padding:8px; border:1px solid #ddd; font-weight:500;">${{item.fr}}</td>`;
                }} else {{
                    html += `<td style="padding:8px; border:1px solid #ddd; font-weight:500;">${{item.en}}</td>`;
                    html += `<td style="padding:8px; border:1px solid #ddd;"><input type="text" class="input-rev" placeholder="Traduire en français..." style="width:95%; padding:6px; border:2px solid #ccc; border-radius:4px; font-size:0.95rem;"/></td>`;
                }}
                
                html += '</tr>';
            }});
            
            html += '</tbody></table>';
            container.innerHTML = html;
        }});
        
        // 2. Gestion des onglets
        function basculerFiche(nomFicheCible) {{
            document.querySelectorAll('.fiche-exercice').forEach(f => f.style.display = 'none');
            document.querySelectorAll('.onglet-btn').forEach(b => b.classList.remove('active'));
            
            const ficheCible = document.querySelector(`.fiche-exercice[data-nom="${{nomFicheCible}}"]`);
            if (ficheCible) ficheCible.style.display = 'block';
            
            const btnActif = Array.from(document.querySelectorAll('.onglet-btn')).find(b => b.textContent === nomFicheCible);
            if (btnActif) btnActif.classList.add('active');
        }}
        
        // 3. Gestion des indices
        function afficherIndice(btn, texteIndice) {{
            const span = btn.nextElementSibling;
            span.textContent = " 💡 Indice : " + texteIndice;
            span.style.display = "inline";
            btn.style.display = "none";
        }}
        
        // 4. Validation Fiche de Révision
        function validerFicheRevision() {{
            document.querySelectorAll('.row-revision').forEach(row => {{
                const mode = row.getAttribute('data-mode');
                const solutionEn = row.getAttribute('data-en').trim().lowerCase ? row.getAttribute('data-en').trim().toLowerCase() : row.getAttribute('data-en').trim();
                const solutionFr = row.getAttribute('data-fr').trim().lowerCase ? row.getAttribute('data-fr').trim().toLowerCase() : row.getAttribute('data-fr').trim();
                
                const input = row.querySelector('.input-rev');
                const reponseUser = input.value.trim().toLowerCase();
                
                let correct = false;
                if (mode === 'en') {{
                    // Gestion des variantes séparées par un slash /
                    const variantes = solutionEn.split('/').map(v => v.trim().toLowerCase());
                    correct = variantes.includes(reponseUser);
                    if(correct) {{
                        input.style.borderColor = "#28a745";
                        input.style.background = "#e8f5e9";
                    }} else {{
                        input.style.borderColor = "#dc3545";
                        input.style.background = "#ffebee";
                        input.value = `${{input.value}} (Attendu: ${{solutionEn}})`;
                    }}
                }} else {{
                    const variantes = solutionFr.split('/').map(v => v.trim().toLowerCase());
                    correct = variantes.includes(reponseUser);
                    if(correct) {{
                        input.style.borderColor = "#28a745";
                        input.style.background = "#e8f5e9";
                    }} else {{
                        input.style.borderColor = "#dc3545";
                        input.style.background = "#ffebee";
                        input.value = `${{input.value}} (Attendu: ${{solutionFr}})`;
                    }}
                }}
            }});
        }}
        
        // 5. Validation Fiches Standards (TRAD / TROU)
        function validerFicheStandard(btnValiderGlobal) {{
            const conteneurFiche = btnValiderGlobal.closest('.fiche-exercice');
            
            conteneurFiche.querySelectorAll('.ex-block').forEach(block => {{
                const correctionBox = block.querySelector('.correction-exo');
                const solutionsBrutes = correctionBox.getAttribute('data-reponses');
                
                // Séparation des cases par des virgules
                const casesSolutions = solutionsBrutes.split(',').map(c => c.trim().toLowerCase());
                const inputs = block.querySelectorAll('.input-custom-exo');
                
                inputs.forEach((input, index) => {{
                    const reponseUser = input.value.trim().toLowerCase();
                    const solutionsPossiblesPourCetteCase = casesSolutions[index] ? casesSolutions[index].split('/').map(s => s.trim()) : [];
                    
                    if (solutionsPossiblesPourCetteCase.includes(reponseUser) && reponseUser !== "") {{
                        input.style.borderColor = "#28a745";
                        input.style.background = "#e8f5e9";
                    }} else {{
                        input.style.borderColor = "#dc3545";
                        input.style.background = "#ffebee";
                    }}
                }});
                
                // Afficher le bloc d'explication / correction
                correctionBox.style.display = "block";
            }});
        }}
        </script>
        
        <style>
        .onglet-btn.active {{ background: #007bff !important; color: white !important; }}
        .input-custom-exo:focus {{ outline: none; box-shadow: 0 0 5px rgba(0,123,255,0.4); }}
        </style>
        """
        
        corps_page_exos = f"""
        <div class="fiche-header-container" style="margin-bottom: 20px;">
            <div class="fiche-titles">
                <p class="intro-text">Module d'entraînement</p>
                <h3 class="fiche-title">Exercices : {titre_propre}</h3>
            </div>
            <a href="./{fichier_html_vocab.name}" class="action-button-link" style="background:#6c757d; color:white; text-decoration:none; padding:8px 15px; border-radius:6px; font-size:0.9rem; font-weight:bold;">← Retour Fiche</a>
        </div>
        
        {onglets_html}
        {fiches_contenu_html}
        {js_code}
        """
        
        breadcrumbs_exos = generer_breadcrumbs(segments_fiche, est_fiche=True) # Utilise la même logique propre
        # Remplacement du nom dans le fil d'Ariane de la page exo pour marquer la nuance
        breadcrumbs_exos = breadcrumbs_exos.replace(titre_propre, f"{titre_propre} (Exos)")
        
        with open(fichier_html_exos, \"w\", encoding=\"utf-8\") as f_out:
            f_out.write(TEMPLATE_CONTENT.format(PAGE_TITLE=f"Exercices - {titre_propre}", MAIN_TITLE="ANGLAIS > Exercices", BREADCRUMBS=breadcrumbs_exos, CONTENT=corps_page_exos))

In [ ]:
# -- B. Génération des Fiches de Vocabulaire & Pages d'Exercices --
for node, data in arborescence.items():
    for fiche in data['fiches']:
        chemin_txt = FILES_DIR / os.path.join(*node) / f"{fiche}.txt"
        segments_fiche = list(node) + [fiche]
        
        # Chemins cibles pour les deux fiches HTML distinctes
        fichier_html_vocab = construire_chemin_page(segments_fiche, est_fiche=True)
        fichier_html_exos = PAGES_DIR / f"{fichier_html_vocab.stem}_exos.html"
        
        titre_propre = fiche.split(';')[0].replace("-", " ").capitalize() if ';' in fiche else fiche.replace("-", " ").capitalize()
        
        # --- Stockages pour structurer la double génération ---
        mots_vocabulaire = []
        exercices_par_onglet = {"Fiche de révisions": []} # L'onglet natif automatique
        
        lecture_exercices = False
        lien_quizlet = None
        
        # --- 1. LECTURE ET PARSING DU FICHIER SOURCE ---
        with open(chemin_txt, "r", encoding="utf-8") as f:
            lignes = f.readlines()
            
        # Détection du lien brut en première ligne
        if lignes and lignes[0].strip().startswith("http"):
            lien_quizlet = lignes[0].strip()
            lignes = lignes[1:]
            
        for line in lignes:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            
            # Repérage de la bascule vers les exercices
            if line == "=== EXERCICES ===":
                lecture_exercices = True
                continue
            
            if not lecture_exercices:
                # Bloc Vocabulaire standard à 7 colonnes
                parts = [p.strip() for p in line.split(';')]
                if len(parts) < 7: continue
                
                mots_vocabulaire.append({
                    "anglais": parts[0], "francais": parts[1],
                    "note": "" if parts[2] == "..." else parts[2],
                    "exemple": "" if parts[3] == "..." else parts[3],
                    "type": convertir_type(parts[4]),
                    "provenance": convertir_provenance(parts[5]),
                    "niveau": parts[6]
                })
            else:
                # Bloc Exercices écrits à la main (7 colonnes)
                parts = [p.strip() for p in line.split(';')]
                if len(parts) < 7: continue
                
                nom_onglet = parts[4] # num_exo sert de nom de fiche d'exercice
                if nom_onglet not in exercices_par_onglet:
                    exercices_par_onglet[nom_onglet] = []
                    
                exercices_par_onglet[nom_onglet].append({
                    "enonce": parts[0], "reponses": parts[1],
                    "indication": "" if parts[2] == "..." else parts[2],
                    "explication": "" if parts[3] == "..." else parts[3],
                    "type": parts[5].upper(), "niveau": parts[6]
                })
        
        # --- 2. ÉCRITURE DU FICHIER DE VOCABULAIRE ---
        tableau_html = '<table class="vocab-table">\n<thead>\n<tr><th>Type</th><th>Anglais</th><th>Français</th><th>Niveau</th><th>Note</th><th>Exemple</th><th>Provenance</th></tr>\n</thead>\n<tbody>\n'
        for m in mots_vocabulaire:
            type_classe = formater_nom_fichier(m['type'])
            niv_classe = m['niveau'].strip().lower()
            
            cell_type = f"<span class='badge badge-type badge-type-{type_classe}'>{m['type']}</span>" if m['type'] != "..." else ""
            cell_level = f"<span class='badge badge-level badge-level-{niv_classe}'>{m['niveau']}</span>" if m['niveau'] != "..." else ""
            cell_prov = f"<span class='badge badge-prov'>{m['provenance']}</span>" if m['provenance'] != "..." else ""
            
            tableau_html += f"<tr><td>{cell_type}</td><td><span class='txt-anglais'>{m['anglais']}</span></td><td><span class='txt-francais'>{m['francais']}</span></td><td>{cell_level}</td><td><span class='txt-note'>{m['note']}</span></td><td><span class='txt-exemple'>{m['exemple']}</span></td><td>{cell_prov}</td></tr>\n"
        tableau_html += "</tbody>\n</table>"
        
        html_bq = ""
        if lien_quizlet:
            html_bq = f'<a href="{lien_quizlet}" target="_blank" class="action-button-link quizlet-btn" title="Ouvrir Quizlet"><img src="../images/quizlet_logo.png" alt="Quizlet"></a>'
            
        corps_page_vocab = f"""
        <div class="fiche-header-container">
            <div class="fiche-titles">
                <p class="intro-text">Fiche de vocabulaire ({len(mots_vocabulaire)} mots)</p>
                <h3 class="fiche-title">{titre_propre}</h3>
            </div>
            <div class="fiche-actions-inline">{html_bq}</div>
        </div>
        
        {tableau_html}
        
        <div style="text-align: center; margin-top: 40px; margin-bottom: 20px;">
            <a href="./{fichier_html_exos.name}" style="padding: 15px 30px; font-size: 1.1rem; display: inline-flex; align-items: center; gap: 10px; background-color: #28a745; color: white; text-decoration: none; border-radius: 8px; font-weight: bold; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
                🎯 S'entraîner avec les exercices
            </a>
        </div>
        """
        
        breadcrumbs_vocab = generer_breadcrumbs(segments_fiche, est_fiche=True)
        html_final_vocab = TEMPLATE_CONTENT.format(PAGE_TITLE=titre_propre, MAIN_TITLE="ANGLAIS > Vocabulaire", BREADCRUMBS=breadcrumbs_vocab, CONTENT=corps_page_vocab)
        with open(fichier_html_vocab, "w", encoding="utf-8") as f_out:
            f_out.write(html_final_vocab)
            
        # --- 3. ÉCRITURE DU FICHIER PARALLÈLE D'EXERCICES ---
        # Menu d'onglets horizontaux nominatifs
        onglets_html = '<div class="onglets-exos-container" style="display: flex; gap: 10px; margin-bottom: 25px; border-bottom: 2px solid #ddd; padding-bottom: 10px;">'
        for i, nom_onglet in enumerate(exercices_par_onglet.keys()):
            classe_active = "active" if i == 0 else ""
            style_supp = "background: #007bff; color: white;" if i == 0 else "background: #f0f0f0; color: #333;"
            onglets_html += f'<button class="onglet-btn {classe_active}" onclick="basculerFiche(this, \'{nom_onglet}\')" style="padding: 10px 20px; border: none; border-radius: 6px; cursor: pointer; font-weight: bold; transition: 0.2s; {style_supp}">{nom_onglet}</button>'
        onglets_html += '</div>'
        
        fiches_contenu_html = '<div class="fiches-exos-wrapper">'
        
        # Onglet automatique A : Fiche de révisions (masquage et tirage géré en JS)
        fiches_contenu_html += f"""
        <div class="fiche-exercice" data-nom="Fiche de révisions" style="display: block;">
            <p class="intro-text" style="margin-bottom: 20px;">Traduisez le mot manquant (L'Anglais ou le Français est masqué de manière aléatoire).</p>
            <div id="container-revision-dynamique"></div>
            <div style="text-align: center; margin-top: 25px;">
                <button onclick="validerFicheRevision()" style="background: #007bff; color: white; padding: 12px 30px; border: none; border-radius: 6px; font-weight: bold; cursor: pointer; font-size: 1rem;">Valider mes réponses</button>
            </div>
        </div>
        """
        
        # Onglets B : Exercices issus du fichier texte (générés s'ils existent)
        for nom_onglet, liste_exos in exercices_par_onglet.items():
            if nom_onglet == "Fiche de révisions": continue
            
            fiches_contenu_html += f'<div class="fiche-exercice" data-nom="{nom_onglet}" style="display: none;">'
            
            for idx, exo in enumerate(liste_exos):
                html_actions = f'<div class="actions-exo" style="margin-top: 8px; display: flex; gap: 10px; align-items: center;">'
                if exo['indication']:
                    html_actions += f'<button onclick="afficherIndice(this, \'{exo["indication"]}\')" style="background:#ffc107; border:none; padding:5px 12px; border-radius:4px; cursor:pointer; font-size:0.85rem; font-weight:bold;">💡 Indice</button>'
                html_actions += f'<span class="txt-indication" style="display:none; color:#775500; font-style:italic; font-size:0.9rem;"></span>'
                html_actions += f'</div>'
                
                html_correction = f'<div class="correction-exo" data-reponses="{exo["reponses"]}" style="margin-top: 12px; display:none; padding:12px; background:#f8f9fa; border-left:4px solid #6c757d; border-radius:0 4px 4px 0;">'
                html_correction += f'<p style="margin:0; font-weight:bold; color:#28a745;">Réponse attendue : {exo["reponses"].replace(",", " , ")}</p>'
                if exo['explication']:
                    html_correction += f'<p style="margin:6px 0 0 0; font-size:0.9rem; color:#555;">💡 <em>{exo["explication"]}</em></p>'
                html_correction += f'</div>'
                
                if exo['type'] == 'TRAD':
                    fiches_contenu_html += f"""
                    <div class="ex-block type-trad" style="background: white; padding: 18px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); margin-bottom: 20px; border-left: 4px solid #007bff;">
                        <p style="margin: 0 0 12px 0; font-size: 1.05rem;"><strong>Question {idx+1} (Traduction) :</strong> Traduire en anglais : <span style="font-style: italic; font-weight: 600; color:#333;">{exo['enonce']}</span></p>
                        <input type="text" class="input-custom-exo" placeholder="Tapez votre traduction ici..." style="width: 100%; max-width: 650px; padding: 10px 14px; border: 2px solid #ccc; border-radius: 6px; font-size: 1rem; box-sizing: border-box;" />
                        {html_actions}
                        {html_correction}
                    </div>
                    """
                elif exo['type'] == 'TROU':
                    enonce_html = exo['enonce'].replace("[…]", '<input type="text" class="input-custom-exo" placeholder="..." style="width: 130px; padding: 5px 10px; border: 2px solid #ccc; border-radius: 4px; text-align: center; font-size: 1rem; margin: 0 5px;" />').replace("[...]", '<input type="text" class="input-custom-exo" placeholder="..." style="width: 130px; padding: 5px 10px; border: 2px solid #ccc; border-radius: 4px; text-align: center; font-size: 1rem; margin: 0 5px;" />')
                    
                    fiches_contenu_html += f"""
                    <div class="ex-block type-trou" style="background: white; padding: 18px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); margin-bottom: 20px; border-left: 4px solid #17a2b8;">
                        <p style="margin: 0 0 12px 0; font-size: 1.05rem;"><strong>Question {idx+1} (Texte à trous) :</strong> Remplissez les espaces vides :</p>
                        <p style="line-height: 2; margin: 5px 0; font-size: 1.05rem; color:#333;">{enonce_html}</p>
                        {html_actions}
                        {html_correction}
                    </div>
                    """
            
            if liste_exos:
                fiches_contenu_html += f"""
                <div style="text-align: center; margin-top: 25px;">
                    <button onclick="validerFicheStandard(this)" style="background: #28a745; color: white; padding: 12px 30px; border: none; border-radius: 6px; font-weight: bold; cursor: pointer; font-size: 1rem;">Valider cette série</button>
                </div>
                """
            fiches_contenu_html += '</div>'
            
        fiches_contenu_html += '</div>'
        
        # --- Injection des scripts JavaScript interactifs ---
        mots_json_bruts = [{"en": m["anglais"], "fr": m["francais"]} for m in mots_vocabulaire]
        import json
        mots_json_string = json.dumps(mots_json_bruts, ensure_ascii=False)
        
        js_code = f"""
        <script>
        const vocabData = {mots_json_string};
        
        // Initialisation de la grille de révision aléatoire
        document.addEventListener("DOMContentLoaded", () => {{
            const container = document.getElementById("container-revision-dynamique");
            let html = '<table class="vocab-table" style="width:100%; border-collapse: collapse; margin-top:15px;">';
            html += '<thead><tr style="background:#f8f9fa;"><th style="padding:12px; border:1px solid #ddd; text-align:left;">Anglais</th><th style="padding:12px; border:1px solid #ddd; text-align:left;">Français</th></tr></thead><tbody>';
            
            vocabData.forEach((item, index) => {{
                const cacherAnglais = Math.random() < 0.5;
                html += `<tr class="row-revision" data-en="${{item.en}}" data-fr="${{item.fr}}" data-mode="${{cacherAnglais ? 'en' : 'fr'}}" style="border-bottom:1px solid #ddd;">`;
                
                if (cacherAnglais) {{
                    html += `<td style="padding:10px; border:1px solid #ddd;"><input type="text" class="input-rev" placeholder="Traduire en anglais..." style="width:98%; padding:8px; border:2px solid #ccc; border-radius:4px; font-size:0.95rem; box-sizing:border-box;"/></td>`;
                    html += `<td style="padding:10px; border:1px solid #ddd; font-weight:500; color:#333;">${{item.fr}}</td>`;
                }} else {{
                    html += `<td style="padding:10px; border:1px solid #ddd; font-weight:500; color:#333;">${{item.en}}</td>`;
                    html += `<td style="padding:10px; border:1px solid #ddd;"><input type="text" class="input-rev" placeholder="Traduire en français..." style="width:98%; padding:8px; border:2px solid #ccc; border-radius:4px; font-size:0.95rem; box-sizing:border-box;"/></td>`;
                }}
                html += '</tr>';
            }});
            html += '</tbody></table>';
            container.innerHTML = html;
        }});
        
        // Navigation par onglets
        function basculerFiche(btnClic, nomFicheCible) {{
            const conteneurPage = btnClic.closest('.fiche-header-container')?.parentElement || document;
            conteneurPage.querySelectorAll('.fiche-exercice').forEach(f => f.style.display = 'none');
            conteneurPage.querySelectorAll('.onglet-btn').forEach(b => {{
                b.style.background = '#f0f0f0';
                b.style.color = '#333';
            }});
            
            const ficheCible = conteneurPage.querySelector(`.fiche-exercice[data-nom="${{nomFicheCible}}"]`);
            if (ficheCible) ficheCible.style.display = 'block';
            
            btnClic.style.background = '#007bff';
            btnClic.style.color = 'white';
        }}
        
        // Affichage des indices
        function afficherIndice(btn, texteIndice) {{
            const span = btn.nextElementSibling;
            span.textContent = " 💡 Indice : " + texteIndice;
            span.style.display = "inline";
            btn.style.display = "none";
        }}
        
        // Validation Onglet de Révisions
        function validerFicheRevision() {{
            document.querySelectorAll('.row-revision').forEach(row => {{
                const mode = row.getAttribute('data-mode');
                const solEn = row.getAttribute('data-en').trim();
                const solFr = row.getAttribute('data-fr').trim();
                const input = row.querySelector('.input-rev');
                const reponseUser = input.value.strip ? input.value.strip().toLowerCase() : input.value.trim().toLowerCase();
                
                const variantes = (mode === 'en' ? solEn : solFr).split('/').map(v => v.trim().toLowerCase());
                
                if (variantes.includes(reponseUser) && reponseUser !== "") {{
                    input.style.borderColor = "#28a745";
                    input.style.background = "#e8f5e9";
                }} else {{
                    input.style.borderColor = "#dc3545";
                    input.style.background = "#ffebee";
                    input.value = `${{input.value}} (Attendu: ${{mode === 'en' ? solEn : solFr}})`;
                }}
            }});
        }}
        
        // Validation Onglets TRAD et TROU
        function validerFicheStandard(btnValiderGlobal) {{
            const conteneurFiche = btnValiderGlobal.closest('.fiche-exercice');
            
            conteneurFiche.querySelectorAll('.ex-block').forEach(block => {{
                const correctionBox = block.querySelector('.correction-exo');
                const casesSolutions = correctionBox.getAttribute('data-reponses').split(',').map(c => c.trim().toLowerCase());
                const inputs = block.querySelectorAll('.input-custom-exo');
                
                inputs.forEach((input, index) => {{
                    const reponseUser = input.value.trim().toLowerCase();
                    const solutionsPossibles = casesSolutions[index] ? casesSolutions[index].split('/').map(s => s.trim()) : [];
                    
                    if (solutionsPossibles.includes(reponseUser) && reponseUser !== "") {{
                        input.style.borderColor = "#28a745";
                        input.style.background = "#e8f5e9";
                    }} else {{
                        input.style.borderColor = "#dc3545";
                        input.style.background = "#ffebee";
                    }}
                }});
                correctionBox.style.display = "block";
            }});
        }}
        </script>
        """
        
        corps_page_exos = f"""
        <div class="fiche-header-container" style="margin-bottom: 20px;">
            <div class="fiche-titles">
                <p class="intro-text">Module d'entraînement</p>
                <h3 class="fiche-title">Exercices : {titre_propre}</h3>
            </div>
            <a href="./{fichier_html_vocab.name}" style="background:#6c757d; color:white; text-decoration:none; padding:10px 18px; border-radius:6px; font-size:0.9rem; font-weight:bold;">← Retour Vocabulaire</a>
        </div>
        
        {onglets_html}
        {fiches_contenu_html}
        {js_code}
        """
        
        breadcrumbs_exos = generer_breadcrumbs(segments_fiche, est_fiche=True).replace(titre_propre, f"{titre_propre} (Exercices)")
        html_final_exos = TEMPLATE_CONTENT.format(PAGE_TITLE=f"Exercices - {titre_propre}", MAIN_TITLE="ANGLAIS > Exercices", BREADCRUMBS=breadcrumbs_exos, CONTENT=corps_page_exos)
        with open(fichier_html_exos, "w", encoding="utf-8") as f_out:
            f_out.write(html_final_exos)